### 举例1:大模型分析工具的调用

In [46]:
# 举例一：大模型分析工具的调用,移动文件或者重命名文件
import os
import dotenv
from langchain_core.messages import HumanMessage

from langchain_openai import ChatOpenAI
from langchain_community.tools import MoveFileTool, BingSearchRun, ReadFileTool

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
llm = ChatOpenAI(
    model='qwen3-vl-flash-2026-01-22',
    temperature=0.6
)

# 获取工具列表
tools = [MoveFileTool(root_dir="/")]

# 绑定工具
llm_with_tools = llm.bind_tools(tools)

# 获取消息列表
messages = [HumanMessage(
    content="请把 /Users/esired/Desktop/ctest/data1.txt文件 移动到/Users/esired/Desktop/ctest/data2.txt文件夹下")]

# 调用大模型
response = llm_with_tools.invoke(
    input=messages,
)
# 4. 获取 ToolCall 并执行
# 注意：在 Agent 环境下，这会自动执行。如果是手动绑定，你需要解析 tool_calls 并执行 tool
tool_call = response.tool_calls[0]
print(f"调用的工具: {tool_call['name']}")
print(f"参数: {tool_call['args']}")

# 手动执行工具 (在实际 Agent 中由框架自动完成)
if tool_call['name'] == 'move_file':
    # 写法1
    # result = tools[0].invoke({
    #     "source_path": tool_call['args'].get('source_path'),
    #     "destination_path": tool_call['args'].get('destination_path')
    # })
    # 写法2:简写
    result = tools[0].invoke(tool_call['args'])   # ✅ 直接传入参数字典
    # print(result)


调用的工具: move_file
参数: {'source_path': '/Users/esired/Desktop/ctest/data1.txt', 'destination_path': '/Users/esired/Desktop/ctest/data2.txt'}


In [62]:
# 举例2:扩展
import os
import dotenv
from langchain_core.messages import HumanMessage

from langchain_openai import ChatOpenAI
from langchain_community.tools import MoveFileTool, BingSearchRun,ReadFileTool

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
llm = ChatOpenAI(
    model='qwen3-vl-flash-2026-01-22',
    temperature=0.6
)

# 获取工具列表
tools = [MoveFileTool(root_dir="/"),ReadFileTool()]

# 绑定工具
llm_with_tools = llm.bind_tools(tools)


# 获取消息列表
messages = [HumanMessage(
    content="读取Mac电脑桌面的/Users/esired/Desktop/json1.txt")]

# 调用大模型
response = llm_with_tools.invoke(
    input=messages,
)

# 注意：在 Agent 环境下，这会自动执行。如果是手动绑定，你需要解析 tool_calls 并执行 tool
'''
    表示取第一个工具调用（通常只有一个）
    或者遍历所有调用
    for tool_call in response.tool_calls:
        # 执行每个工具
        ...
'''
tool_call = response.tool_calls[0]
print(f"调用的工具: {tool_call['name']}")
print(f"参数: {tool_call['args']}")

if tool_call['name'] == 'read_file':
    res=tools[1].invoke(tool_call['args'])
    print("文件内容如下：")
    print(res)

调用的工具: read_file
参数: {'file_path': '~/Desktop/json1.txt'}
文件内容如下：
Error: no such file or directory: ~/Desktop/json1.txt
